# 📓 Notebook 03 — Embeddings & Semantic Search---**Phase 1 · LLM Fundamentals**> Embeddings are the **backbone of RAG, search, recommendations, and clustering**. Master them and you unlock everything.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Vector spaces | How meaning becomes math || Embedding models | Multi-model: OpenAI, Gemini, sentence-transformers || Distance metrics | Cosine vs dot product vs euclidean — when to use which || Semantic search | Find by meaning, not keywords || Practical patterns | Batching, caching, dimensionality |### 🏗️ Build: Multi-Model Semantic Search Engine

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode, DEFAULT_EMBEDDING

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. What Are Embeddings?### The Core IdeaAn **embedding** is a dense vector (list of floats) that represents the **meaning** of text in a high-dimensional space.```"The cat sat on the mat"  →  [0.023, -0.156, 0.891, ..., 0.042]   # 1536 dimensions```### Why This Matters- **Similar meanings → nearby vectors** → enables semantic search- **Different from keywords** → "automobile" ≈ "car" even though different words- Powers: search, RAG, recommendations, clustering, anomaly detection### How Embeddings Work (Interview Depth)```                     Text Input                         ↓               ┌─────────────────┐               │  Transformer    │   (BERT, GPT, etc.)               │  Encoder        │               └────────┬────────┘                         ↓              [0.02, -0.15, 0.89, ...]   ← Dense vector (1536-D)                         ↓               Meaning in vector space```| Property | Detail ||----------|--------|| Dimensions | 256–3072 depending on model || Values | Floating point numbers, typically -1 to 1 (normalized) || Key insight | Cosine similarity between vectors ≈ semantic similarity || Training | Learned from massive text corpora (contrastive learning) |

### 🗺️ Visualizing the Embedding Space

![Embeddings: Meaning as Location in Space](images/vector_space_embeddings.png)

#### 💡 Intuition for Juniors

Imagine a **map of meaning** where every piece of text has a location:
- 📍 "cat" and "kitten" are **neighbors** — they mean similar things.
- 📍 "cat" and "pizza" are in **different neighborhoods** — totally unrelated.
- 📍 "Python (programming)" and "Python (snake)" are **closer than you'd expect** — the model captures context!

The diagram shows this in 2D, but real embeddings use **768–3072 dimensions**. More dimensions = finer distinctions between meanings.

> **Mental model:** Embeddings are like GPS coordinates for meaning. Similarity search = "find the nearest points."

### Vectors from Scratch -- A Complete Beginner Guide

#### What IS a Vector?

A vector is just **a list of numbers**. That is it.

```
[3, 4]           <-- a 2D vector (two numbers)
[1, 0, -1]       <-- a 3D vector (three numbers)
[0.02, -0.15, 0.89, ..., 0.04]  <-- a 1536D vector (embedding!)
```

You can think of a vector as **an arrow** pointing from the origin to a point in space.
In 2D: `[3, 4]` means go 3 right and 4 up -- that is where the arrow points.

---

#### Why Use Vectors for Meaning?

**The key insight:** If we can turn words into vectors where **similar words get similar vectors**, then we can use math to find similar content!

Imagine a simple 2D example:
```
"happy"   -> [0.9,  0.1]    (high positive-emotion, low formality)
"joyful"  -> [0.85, 0.15]   (similar! almost same direction)
"sad"     -> [-0.8, 0.1]    (opposite direction! negative emotion)
"formal"  -> [0.1,  0.9]    (different direction entirely)
```

Real embeddings use 768-3072 dimensions instead of 2, capturing hundreds of aspects of meaning.

---

#### Why Not Just Use Keywords?

Keywords fail because:
```
"automobile" and "car" -> same meaning, ZERO keyword overlap!
"bank" (river) and "bank" (money) -> same keyword, DIFFERENT meanings!
```

Vectors solve both problems because the model learns meaning, not spelling.

In [ ]:
from IPython.display import HTMLHTML("""<div id="cluster-demo" style="font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#0f0c29,#1a1a2e);padding:24px;border-radius:16px;color:#e0e0e0;max-width:650px;box-shadow:0 8px 32px rgba(0,0,0,0.4);">  <h3 style="margin:0 0 4px;color:#22c55e;font-size:18px;">Watch Embeddings Cluster by Meaning</h3>  <p style="margin:0 0 12px;font-size:13px;color:#888;">Words start at random positions, then drift to their correct clusters</p>  <canvas id="clusterCanvas" width="620" height="320" style="border-radius:12px;background:#0a0a1a;display:block;margin:0 auto;"></canvas>  <div style="display:flex;gap:20px;justify-content:center;margin-top:12px;">    <span style="color:#f472b6;">Animals</span>    <span style="color:#00d2ff;">Technology</span>    <span style="color:#22c55e;">Food</span>  </div>  <button id="clusterBtn" onclick="window._startCluster()" style="display:block;margin:12px auto 0;padding:8px 24px;background:linear-gradient(135deg,#22c55e,#16a34a);color:#fff;border:none;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;">Animate Clustering</button></div><script>(function(){  var canvas=document.getElementById('clusterCanvas'),ctx=canvas.getContext('2d');  var W=canvas.width,H=canvas.height;  var words=[    {t:'cat',c:'#f472b6',tx:120,ty:80},{t:'dog',c:'#f472b6',tx:140,ty:110},{t:'puppy',c:'#f472b6',tx:100,ty:130},{t:'kitten',c:'#f472b6',tx:160,ty:95},    {t:'python',c:'#00d2ff',tx:350,ty:200},{t:'code',c:'#00d2ff',tx:380,ty:230},{t:'algorithm',c:'#00d2ff',tx:320,ty:240},{t:'GPU',c:'#00d2ff',tx:370,ty:175},    {t:'pizza',c:'#22c55e',tx:500,ty:80},{t:'pasta',c:'#22c55e',tx:520,ty:110},{t:'sushi',c:'#22c55e',tx:480,ty:60},{t:'burger',c:'#22c55e',tx:530,ty:90}  ];  words.forEach(function(w){w.x=50+Math.random()*(W-100);w.y=50+Math.random()*(H-100);});  var animating=false,progress=0;  function draw(){    ctx.clearRect(0,0,W,H);    if(progress>0.3){      var alpha=Math.min((progress-0.3)*1.5,0.15);      [[130,100,'#f472b6'],[355,210,'#00d2ff'],[510,85,'#22c55e']].forEach(function(a){        ctx.fillStyle=a[2].slice(0,7)+(Math.floor(alpha*255).toString(16).padStart(2,'0'));        ctx.beginPath();ctx.arc(a[0],a[1],55,0,Math.PI*2);ctx.fill();      });    }    words.forEach(function(w){      ctx.fillStyle=w.c;ctx.beginPath();ctx.arc(w.x,w.y,6,0,Math.PI*2);ctx.fill();      ctx.shadowBlur=8;ctx.shadowColor=w.c;ctx.fill();ctx.shadowBlur=0;      ctx.fillStyle='#fff';ctx.font='11px sans-serif';ctx.fillText(w.t,w.x+9,w.y+4);    });  }  function step(){    if(!animating)return;    progress=Math.min(progress+0.008,1);    var ease=progress<0.5?2*progress*progress:1-Math.pow(-2*progress+2,2)/2;    words.forEach(function(w){w.x+=(w.tx-w.x)*0.03*ease;w.y+=(w.ty-w.y)*0.03*ease;});    draw();    if(progress<1)requestAnimationFrame(step);    else{document.getElementById('clusterBtn').textContent='Reset and Replay';animating=false;}  }  window._startCluster=function(){    if(progress>=1){words.forEach(function(w){w.x=50+Math.random()*(W-100);w.y=50+Math.random()*(H-100);});progress=0;}    animating=true;document.getElementById('clusterBtn').textContent='Clustering...';    step();  };  draw();})();</script>""")

In [ ]:
def get_embedding(text, model=None):    """Get embedding vector for text using litellm."""    model = model or DEFAULT_EMBEDDING    response = litellm.embedding(model=model, input=[text])    return response.data[0]["embedding"]def get_embeddings_batch(texts, model=None):    """Batch embed multiple texts (more efficient)."""    model = model or DEFAULT_EMBEDDING    response = litellm.embedding(model=model, input=texts)    return [d["embedding"] for d in response.data]# Get our first embeddingtext = "Machine learning is a subset of artificial intelligence."embedding = get_embedding(text)print(f"📐 Embedding for: \"{text}\"")print(f"   Dimensions: {len(embedding)}")print(f"   First 10 values: {[round(v, 4) for v in embedding[:10]]}")print(f"   Min: {min(embedding):.4f}, Max: {max(embedding):.4f}")print(f"   Mean: {np.mean(embedding):.4f}, Std: {np.std(embedding):.4f}")print(f"   L2 Norm: {np.linalg.norm(embedding):.4f}")

---## 3. Distance Metrics — Measuring Similarity### The Three Main Metrics| Metric | Formula | Range | Best For ||--------|---------|-------|----------|| **Cosine Similarity** | cos(θ) = A·B / (\|A\|·\|B\|) | [-1, 1] | Normalized embeddings (most common) || **Dot Product** | A·B | (-∞, +∞) | When magnitude matters (popularity) || **Euclidean Distance** | √Σ(aᵢ-bᵢ)² | [0, +∞) | When absolute distance matters |> **Interview Critical:** Most embedding models output *normalized* vectors (L2 norm ≈ 1).  > For normalized vectors, cosine similarity = dot product. This is why FAISS uses inner product by default.

### The Dot Product -- What It Actually Calculates

#### Step 1: What is a Dot Product?

The dot product takes two vectors and returns **one number** that tells you how much they agree.

```
A = [3, 4]     B = [4, 3]

dot(A, B) = (3 x 4) + (4 x 3)
          =   12   +   12
          =   24
```

**Recipe:** Multiply matching positions, then add everything up.

---

#### Why Does This Measure Similarity?

Think of each dimension as a **question**:
- Dimension 1 = How much is this about technology? (scale: -5 to +5)
- Dimension 2 = How positive is the sentiment? (scale: -5 to +5)

```
"I love Python!"   -> [4, 3]    (techy AND positive)
"Great JavaScript"  -> [3, 4]    (techy AND positive)
"Terrible weather"  -> [-1, -4]  (not techy AND negative)
```

When two vectors **agree** on a dimension (both positive or both negative), their product is **positive** -- adds to similarity.
When they **disagree** (one positive, one negative), the product is **negative** -- subtracts from similarity.

```
dot(Python, JavaScript)  = (4x3) + (3x4) = 12 + 12 = +24  <-- Similar!
dot(Python, Weather)     = (4x-1) + (3x-4) = -4 + -12 = -16  <-- Opposite!
```

---

#### The Magnitude (Length)

The magnitude is **how long the arrow is**:

```
|A| = sqrt(3^2 + 4^2) = sqrt(9 + 16) = sqrt(25) = 5
```

This is just the Pythagorean theorem! For a right triangle with sides 3 and 4, the hypotenuse is 5.

In [ ]:
from IPython.display import HTMLHTML("""<div id="cosine-demo" style="font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#0f0c29,#1a1a2e);padding:24px;border-radius:16px;color:#e0e0e0;max-width:650px;box-shadow:0 8px 32px rgba(0,0,0,0.4);">  <h3 style="margin:0 0 4px;color:#a78bfa;font-size:18px;">Interactive: Drag the Vector!</h3>  <p style="margin:0 0 12px;font-size:13px;color:#888;">Move Vector B (orange) around to see how cosine similarity changes</p>  <div style="display:flex;gap:16px;align-items:flex-start;">    <canvas id="cosCanvas" width="280" height="280" style="border-radius:12px;background:#0a0a1a;cursor:crosshair;"></canvas>    <div style="flex:1;">      <div style="background:rgba(0,210,255,0.08);padding:12px;border-radius:10px;margin-bottom:10px;">        <div style="font-size:12px;color:#6ec6ff;">Vector A (fixed)</div>        <div style="font-size:20px;font-weight:bold;color:#00d2ff;" id="vecA">[3, 4]</div>      </div>      <div style="background:rgba(251,146,60,0.08);padding:12px;border-radius:10px;margin-bottom:10px;">        <div style="font-size:12px;color:#fb923c;">Vector B (drag me!)</div>        <div style="font-size:20px;font-weight:bold;color:#fb923c;" id="vecB">[4, 3]</div>      </div>      <div style="background:rgba(167,139,250,0.08);padding:12px;border-radius:10px;margin-bottom:10px;">        <div style="font-size:12px;color:#a78bfa;">Cosine Similarity</div>        <div style="font-size:32px;font-weight:bold;color:#a78bfa;" id="cosVal">0.96</div>        <div style="font-size:12px;color:#888;margin-top:2px;" id="cosAngle">angle = 16.3 deg</div>      </div>      <div id="cosBar" style="height:8px;border-radius:4px;background:linear-gradient(90deg,#ef4444,#888,#22c55e);margin-bottom:4px;position:relative;">        <div id="cosMarker" style="position:absolute;top:-4px;width:16px;height:16px;border-radius:50%;background:#fff;border:2px solid #a78bfa;transition:left 0.15s;"></div>      </div>      <div style="display:flex;justify-content:space-between;font-size:10px;color:#666;"><span>-1 Opposite</span><span>0 Unrelated</span><span>+1 Identical</span></div>      <div id="cosInsight" style="margin-top:10px;padding:8px;background:rgba(167,139,250,0.06);border-radius:6px;font-size:12px;color:#aaa;border-left:3px solid #a78bfa;"></div>    </div>  </div></div><script>(function(){  var canvas = document.getElementById('cosCanvas');  var ctx = canvas.getContext('2d');  var W = canvas.width, H = canvas.height;  var cx = W/2, cy = H/2, scale = 28;  var bx = 4, by = 3;  var ax = 3, ay = 4;  var dragging = false;  function draw(){    ctx.clearRect(0,0,W,H);    ctx.strokeStyle='#222';ctx.lineWidth=1;    for(var i=-5;i<=5;i++){ctx.beginPath();ctx.moveTo(cx+i*scale,0);ctx.lineTo(cx+i*scale,H);ctx.stroke();ctx.beginPath();ctx.moveTo(0,cy-i*scale);ctx.lineTo(W,cy-i*scale);ctx.stroke();}    ctx.strokeStyle='#444';ctx.lineWidth=1.5;ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(W,cy);ctx.stroke();ctx.beginPath();ctx.moveTo(cx,0);ctx.lineTo(cx,H);ctx.stroke();    function arrow(x,y,color,lbl){      ctx.strokeStyle=color;ctx.lineWidth=3;ctx.beginPath();ctx.moveTo(cx,cy);ctx.lineTo(cx+x*scale,cy-y*scale);ctx.stroke();      var len=Math.sqrt(x*x+y*y)*scale;var ux=(x*scale)/len;var uy=(-y*scale)/len;      ctx.fillStyle=color;ctx.beginPath();ctx.moveTo(cx+x*scale,cy-y*scale);ctx.lineTo(cx+x*scale-ux*10+uy*5,cy-y*scale-uy*10-ux*5);ctx.lineTo(cx+x*scale-ux*10-uy*5,cy-y*scale-uy*10+ux*5);ctx.fill();      ctx.fillStyle=color;ctx.font='bold 12px sans-serif';ctx.fillText(lbl,cx+x*scale+5,cy-y*scale-5);    }    arrow(ax,ay,'#00d2ff','A');arrow(bx,by,'#fb923c','B');    ctx.fillStyle='#00d2ff';ctx.beginPath();ctx.arc(cx+ax*scale,cy-ay*scale,5,0,Math.PI*2);ctx.fill();    ctx.fillStyle='#fb923c';ctx.beginPath();ctx.arc(cx+bx*scale,cy-by*scale,7,0,Math.PI*2);ctx.fill();  }  function updateInfo(){    var dot=ax*bx+ay*by;var magA=Math.sqrt(ax*ax+ay*ay);var magB=Math.sqrt(bx*bx+by*by);    var cos=magB===0?0:dot/(magA*magB);var angle=Math.acos(Math.max(-1,Math.min(1,cos)))*180/Math.PI;    document.getElementById('vecB').textContent='['+bx.toFixed(1)+', '+by.toFixed(1)+']';    var el=document.getElementById('cosVal');el.textContent=cos.toFixed(3);    el.style.color=cos>0.7?'#22c55e':cos>0.3?'#eab308':cos>-0.3?'#888':'#ef4444';    document.getElementById('cosAngle').textContent='angle = '+angle.toFixed(1)+' deg';    var pct=((cos+1)/2*100);document.getElementById('cosMarker').style.left='calc('+pct+'% - 8px)';    var ins=document.getElementById('cosInsight');    if(cos>0.9)ins.innerHTML='<b>Very similar!</b> These vectors point almost the same direction.';    else if(cos>0.5)ins.innerHTML='<b>Somewhat similar.</b> There is meaningful overlap in direction.';    else if(cos>-0.1)ins.innerHTML='<b>Unrelated.</b> These vectors are nearly perpendicular (90 deg).';    else ins.innerHTML='<b>Opposite!</b> These vectors point in opposing directions.';  }  canvas.addEventListener('mousedown',function(e){dragging=true;});  canvas.addEventListener('mouseup',function(){dragging=false;});  canvas.addEventListener('mouseleave',function(){dragging=false;});  canvas.addEventListener('mousemove',function(e){    if(!dragging)return;var r=canvas.getBoundingClientRect();    bx=((e.clientX-r.left)-cx)/scale;by=(cy-(e.clientY-r.top))/scale;    bx=Math.max(-4.5,Math.min(4.5,bx));by=Math.max(-4.5,Math.min(4.5,by));    draw();updateInfo();  });  canvas.addEventListener('touchstart',function(e){e.preventDefault();dragging=true;},{passive:false});  canvas.addEventListener('touchend',function(){dragging=false;});  canvas.addEventListener('touchmove',function(e){    e.preventDefault();if(!dragging)return;var t=e.touches[0];var r=canvas.getBoundingClientRect();    bx=((t.clientX-r.left)-cx)/scale;by=(cy-(t.clientY-r.top))/scale;    bx=Math.max(-4.5,Math.min(4.5,bx));by=Math.max(-4.5,Math.min(4.5,by));    draw();updateInfo();  },{passive:false});  draw();updateInfo();})();</script>""")

### Why Cosine Similarity? Why Not Just Distance?

#### Problem with Euclidean Distance

Euclidean distance measures **how far apart** two points are (like a ruler).
But consider this:

```
Short review:  'Great movie!'                      -> [0.3, 0.4]   (small)
Long review:   'Great movie! Amazing! Brilliant!'  -> [3.0, 4.0]   (big)
Negative:      'Terrible film.'                    -> [-0.3, -0.4] (opposite)
```

**Euclidean distances:**
```
Short <-> Long     = 4.5  (FAR apart!)
Short <-> Negative = 1.0  (close?!)
```

The negative review is 'closer' to the short positive review than the long positive one!
Euclidean distance is fooled by **magnitude** (length).

---

#### Cosine Similarity Ignores Length

Cosine only cares about **direction** (the angle), not length:
```
cosine(Short, Long)     = 1.0   <-- Same direction! Both positive reviews
cosine(Short, Negative) = -1.0  <-- Opposite direction! Positive vs negative
```

**That is why cosine is the standard for embeddings** -- a short text and a long text about the same topic should be similar.

---

#### When IS Euclidean Distance Useful?

| Metric | Cares About | Best For |
|--------|-------------|----------|
| **Cosine** | Direction only | Text similarity, search, RAG |
| **Euclidean** | Direction AND magnitude | Anomaly detection, clustering |
| **Dot Product** | Direction AND magnitude | When magnitude = importance |

> **Pro tip:** Most embedding APIs return **normalized** vectors (magnitude = 1). When all vectors have the same length, cosine = dot product. That is why many vector DBs use dot product internally -- it is faster.

### Normalization -- Why Make All Vectors Length 1?

**Normalizing** a vector means scaling it so its length becomes exactly 1.

```
Original:   [3, 4]     -> length = sqrt(9+16) = 5
Normalized: [3/5, 4/5] = [0.6, 0.8]  -> length = sqrt(0.36+0.64) = 1.0
```

**Recipe:** Divide each number by the vector's length.

#### Why Do This?

1. **Makes cosine = dot product** -- dot product is faster (just multiply and add, skip the division)
2. **Removes magnitude bias** -- long documents do not unfairly dominate short ones
3. **Most APIs do it for you** -- OpenAI, Gemini, and sentence-transformers all return pre-normalized vectors

#### Quick Check: Is My Vector Normalized?
```python
import numpy as np
norm = np.linalg.norm(embedding)
print(f'Norm: {norm:.4f}')  # Should be ~1.0000
```

### 📐 Cosine Similarity — Visual Explanation

![Cosine Similarity: Measuring Direction, Not Distance](images/cosine_similarity_explained.png)

#### 💡 Intuition: It's All About Direction

Cosine similarity measures whether two vectors **point in the same direction**, regardless of their length:
- Two arrows pointing the **same way** → cosine = 1.0 (identical meaning)
- Two arrows at **right angles** → cosine = 0.0 (completely unrelated)
- Two arrows pointing **opposite ways** → cosine = -1.0 (opposite meaning)

#### 📝 Step-by-Step Worked Example

Let's walk through a concrete calculation with simple 2D vectors:

**Given:** Vector A = [3, 4] and Vector B = [4, 3]

**Step 1: Calculate the dot product (A · B)**
```
A · B = (3 × 4) + (4 × 3) = 12 + 12 = 24
```
*The dot product multiplies matching dimensions and sums them up.*

**Step 2: Calculate the magnitude (length) of each vector**
```
|A| = √(3² + 4²) = √(9 + 16) = √25 = 5
|B| = √(4² + 3²) = √(16 + 9) = √25 = 5
```
*The magnitude is like measuring the arrow's length with a ruler.*

**Step 3: Apply the formula**
```
cosine_similarity = dot_product / (|A| × |B|)
                  = 24 / (5 × 5)
                  = 24 / 25
                  = 0.96
```

**Step 4: Interpret the result**
- 0.96 is very close to 1.0 → these vectors are **very similar**!
- They almost point in the same direction (small angle between them).

**More examples to build intuition:**

| Vectors | Cosine | Meaning |
|---------|--------|---------|
| [3, 4] vs [3, 4] | 1.00 | Identical — same direction |
| [3, 4] vs [4, 3] | 0.96 | Very similar — close angle |
| [1, 0] vs [0, 1] | 0.00 | Perpendicular — unrelated |
| [3, 4] vs [-3, -4] | -1.00 | Opposite — reversed meaning |

> **Why cosine and not just distance?** Because embeddings can have different magnitudes.
> A long document and a short summary of it might have very different *lengths* but point in the *same direction*.
> Cosine ignores length and focuses on meaning.

In [ ]:
def cosine_similarity(a, b):    """Cosine similarity between two vectors."""    a, b = np.array(a), np.array(b)    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))def dot_product(a, b):    """Dot product between two vectors."""    return np.dot(np.array(a), np.array(b))def euclidean_distance(a, b):    """Euclidean distance between two vectors."""    return np.linalg.norm(np.array(a) - np.array(b))# Compare semantically similar vs different textspairs = [    ("The cat sat on the mat", "A feline rested on a rug"),            # Same meaning    ("The cat sat on the mat", "The stock market crashed today"),       # Very different    ("Python is a programming language", "Python is a type of snake"), # Ambiguous!    ("I love machine learning", "I enjoy AI and deep learning"),       # Similar domain    ("The weather is sunny today", "It's a beautiful bright day"),     # Paraphrase]print("📊 Similarity Analysis")print("=" * 80)# Batch embed all unique textsall_texts = list(set([t for pair in pairs for t in pair]))all_embeddings = get_embeddings_batch(all_texts)embed_map = dict(zip(all_texts, all_embeddings))for text_a, text_b in pairs:    emb_a = embed_map[text_a]    emb_b = embed_map[text_b]    cos = cosine_similarity(emb_a, emb_b)    dot = dot_product(emb_a, emb_b)    euc = euclidean_distance(emb_a, emb_b)        bar = "█" * int(cos * 30) if cos > 0 else ""    print(f"\n  \"{text_a}\"")    print(f"  \"{text_b}\"")    print(f"  Cosine: {cos:.4f} {bar}")    print(f"  Dot:    {dot:.4f}")    print(f"  Eucl:   {euc:.4f}")

### 💡 When to Use Which Metric| Scenario | Best Metric | Why ||----------|-------------|-----|| Semantic text search | Cosine similarity | Direction matters, not magnitude || Document with popularity scores | Dot product | Longer docs = higher magnitude = boosted || Anomaly detection | Euclidean distance | Absolute distance from cluster center || Normalized vectors (most APIs) | Cosine ≡ Dot product | They're identical when ‖v‖=1 |> **Interview Tip:** *"Are cosine similarity and dot product ever the same?"*  > Yes! When vectors are L2-normalized (unit vectors), cosine = dot product. Most embedding APIs return normalized vectors.

---## 4. Embedding Models — Comparison| Model | Provider | Dimensions | Speed | Quality | Cost ||-------|----------|------------|-------|---------|------|| `text-embedding-3-small` | OpenAI | 1536 | Fast | Good | $0.02/1M tokens || `text-embedding-3-large` | OpenAI | 3072 | Medium | Best | $0.13/1M tokens || `text-embedding-004` | Google | 768 | Fast | Good | Free tier available || `all-MiniLM-L6-v2` | HuggingFace | 384 | Very fast | Good | Free (local) || `all-mpnet-base-v2` | HuggingFace | 768 | Medium | Better | Free (local) |> **Interview Tip:** *"How do you choose an embedding model?"*  > Consider: quality needs, latency requirements, cost budget, deployment constraints (cloud vs local).  > Start with `text-embedding-3-small` for most use cases. Use local models for privacy/offline.

In [ ]:
# Compare embedding dimensions and norms across models# (Uses your default model — you can change DEFAULT_EMBEDDING in .env)test_text = "Artificial intelligence is transforming the way we build software."emb = get_embedding(test_text)print(f"📐 Embedding Model Analysis: {DEFAULT_EMBEDDING}")print(f"   Dimensions: {len(emb)}")print(f"   L2 Norm: {np.linalg.norm(emb):.6f}")print(f"   Is normalized: {'✅ Yes' if abs(np.linalg.norm(emb) - 1.0) < 0.01 else '❌ No'}")print(f"   Memory per vector: {len(emb) * 4} bytes (float32)")print(f"   Memory for 1M vectors: {len(emb) * 4 * 1_000_000 / (1024**3):.2f} GB")

---## 5. Practical Patterns### 5a. Batch ProcessingAlways embed in batches — single calls are 10-50x slower per item.

In [ ]:
# Batch vs single embedding performance comparisontexts = [    "Machine learning automates data analysis",    "Deep learning uses neural networks",     "Natural language processing handles text",    "Computer vision processes images",    "Reinforcement learning trains through rewards",    "Transfer learning reuses pretrained models",    "Federated learning trains across devices",    "AutoML automates model selection",]# Method 1: One at a time (slow)start = time.time()single_embs = [get_embedding(t) for t in texts]single_time = time.time() - start# Method 2: Batch (fast)start = time.time()batch_embs = get_embeddings_batch(texts)batch_time = time.time() - startprint(f"⏱️ Performance Comparison ({len(texts)} texts)")print(f"   Single calls: {single_time:.2f}s ({single_time/len(texts)*1000:.0f}ms per text)")print(f"   Batch call:   {batch_time:.2f}s ({batch_time/len(texts)*1000:.0f}ms per text)")print(f"   Speedup:      {single_time/batch_time:.1f}x faster with batching")

### 5b. Embedding VisualizationHigh-dimensional vectors can be reduced to 2D/3D using **t-SNE** or **PCA** for visualization.

In [ ]:
from sklearn.decomposition import PCAimport numpy as np# Embed categorized textscategories = {    "🐱 Animals": ["The cat hunts mice", "Dogs are loyal pets", "Birds fly south in winter", "Elephants never forget"],    "💻 Tech": ["Neural networks learn patterns", "Python is popular for AI", "GPUs accelerate training", "Cloud computing scales easily"],    "🍕 Food": ["Pizza is popular worldwide", "Sushi requires fresh fish", "Pasta comes from Italy", "Tacos are Mexican cuisine"],}all_texts = []all_labels = []for cat, texts in categories.items():    all_texts.extend(texts)    all_labels.extend([cat] * len(texts))embeddings = get_embeddings_batch(all_texts)emb_matrix = np.array(embeddings)# Reduce to 2D with PCApca = PCA(n_components=2)coords = pca.fit_transform(emb_matrix)print("📊 2D Embedding Visualization (PCA)")print(f"   Explained variance: {pca.explained_variance_ratio_.sum():.1%}")print("=" * 60)# Text-based scatter plotfor i, (label, text) in enumerate(zip(all_labels, all_texts)):    x, y = coords[i]    print(f"  {label} ({x:+.2f}, {y:+.2f}) → \"{text}\"")# Show within-cluster vs between-cluster distancesprint("\n📏 Cluster Distances:")for cat_a in categories:    for cat_b in categories:        if cat_a >= cat_b:            continue        idxs_a = [i for i, l in enumerate(all_labels) if l == cat_a]        idxs_b = [i for i, l in enumerate(all_labels) if l == cat_b]        dists = [cosine_similarity(embeddings[a], embeddings[b]) for a in idxs_a for b in idxs_b]        print(f"  {cat_a} ↔ {cat_b}: avg cosine = {np.mean(dists):.4f}")

---## 📝 Interview Quiz — Embeddings### Q1: What is an embedding? How is it different from one-hot encoding?<details><summary>💡 Answer</summary>**Embedding:** A dense, learned vector representation where similar items are close in vector space.- Dense: Every dimension has a meaningful value- Learned: Trained from data (not manually designed)- Fixed size regardless of vocabulary**One-hot encoding:** A sparse binary vector where one position is 1.- Sparse: Mostly zeros- No similarity: every pair is equidistant - Size = vocabulary size (can be millions)**Key difference:** Embeddings capture *semantic similarity* — "king" and "queen" are close, but one-hot treats them as completely unrelated.</details>### Q2: Explain cosine similarity. When does it equal dot product?<details><summary>💡 Answer</summary>**Cosine similarity** = cos(θ) = (A·B) / (‖A‖ × ‖B‖)- Measures the *angle* between two vectors- Range: [-1, 1] (1 = identical direction, 0 = orthogonal, -1 = opposite)- Ignores vector magnitude (length)**Equals dot product when:** vectors are L2-normalized (unit vectors, ‖v‖=1).- Most embedding APIs return normalized vectors- So in practice, cosine similarity = dot product for API embeddings- FAISS `IndexFlatIP` (inner product) is equivalent to cosine search for normalized vectors</details>### Q3: How would you choose an embedding model for production?<details><summary>💡 Answer</summary>Decision factors:1. **Quality**: Check MTEB leaderboard for task-specific benchmarks2. **Latency**: Local models (sentence-transformers) < API models for high QPS3. **Cost**: OpenAI charges per token, local models are free after setup4. **Privacy**: Sensitive data → local models (no data leaves your servers)5. **Dimensions**: More dims = better quality but more storage/compute6. **Context length**: Some models handle longer texts better**Rule of thumb:** Start with `text-embedding-3-small` (OpenAI) or `all-MiniLM-L6-v2` (local), benchmark, then upgrade if needed.</details>### Q4: You have 10 million documents to embed. How would you do it efficiently?<details><summary>💡 Answer</summary>1. **Batch processing**: Send 100-2000 texts per API call2. **Parallel requests**: Use asyncio or thread pools (respect rate limits)3. **Cache results**: Store embeddings alongside documents (avoid re-computation)4. **Chunk large documents**: Most models have token limits (8192 for OpenAI)5. **Use cheaper models**: `text-embedding-3-small` over `large` unless quality demands it6. **Local models**: For high volume, run sentence-transformers on GPU — no API costs7. **Incremental updates**: Only embed new/changed documents, not the entire corpus**Cost estimate:** 10M docs × 500 tokens avg × $0.02/1M tokens = $100 with OpenAI small model.</details>### Q5: What is the "curse of dimensionality"? How does it affect similarity search?<details><summary>💡 Answer</summary>In very high-dimensional spaces (1000+):- All points become roughly **equidistant** from each other- **Nearest neighbor** becomes less meaningful- The difference between nearest and farthest neighbor shrinks**Impact on similarity search:**- Exact nearest-neighbor search becomes slow (must check every vector)- Approximate methods (HNSW, IVF) become necessary- Dimensionality reduction (PCA) can help but loses information**Why embeddings still work:** Even though raw distances converge, the **relative ordering** is preserved. The most similar document is still returned first — the absolute scores just become less interpretable.</details>### Q6: Can you use the same embedding model for different languages?<details><summary>💡 Answer</summary>**Depends on the model:**- **OpenAI's text-embedding-3**: Multilingual, works across 100+ languages in the same vector space- **sentence-transformers/multilingual models**: Specifically trained for cross-lingual similarity- **English-only models**: Will produce poor results for other languages**Cross-lingual search:** Multilingual models let you query in English and find Hindi documents — the embeddings share the same semantic space regardless of language.**Best practice:** Test with your specific languages before committing. Use multilingual benchmarks (MTEB multilingual subset).</details>

---## 🏗️ BUILD: Multi-Model Semantic Search EngineA complete in-memory semantic search engine with:- Multi-model embedding support- Cosine similarity ranking- Result explanation- Performance metrics

In [ ]:
class SemanticSearchEngine:    """In-memory semantic search engine with multi-model support."""        def __init__(self, embedding_model=None):        self.model = embedding_model or DEFAULT_EMBEDDING        self.documents = []      # Original texts        self.metadata = []       # Optional metadata per doc        self.embeddings = []     # Embedding vectors        self.index_built = False        def add_documents(self, texts, metadata_list=None):        """Add documents to the search index."""        # Batch embed        new_embeddings = get_embeddings_batch(texts, model=self.model)                self.documents.extend(texts)        self.embeddings.extend(new_embeddings)        if metadata_list:            self.metadata.extend(metadata_list)        else:            self.metadata.extend([{} for _ in texts])                self.index_built = True        print(f"  📥 Added {len(texts)} documents (total: {len(self.documents)})")        def search(self, query, top_k=5):        """Search for documents most similar to the query."""        assert self.index_built, "No documents indexed yet!"                # Embed query        query_emb = get_embedding(query, model=self.model)                # Calculate similarities        similarities = []        for i, doc_emb in enumerate(self.embeddings):            sim = cosine_similarity(query_emb, doc_emb)            similarities.append((i, sim))                # Sort by similarity (descending)        similarities.sort(key=lambda x: x[1], reverse=True)                # Return top_k results        results = []        for idx, sim in similarities[:top_k]:            results.append({                "rank": len(results) + 1,                "text": self.documents[idx],                "score": round(sim, 4),                "metadata": self.metadata[idx],            })        return results        def stats(self):        """Print index statistics."""        if not self.embeddings:            print("  Empty index")            return        emb_array = np.array(self.embeddings)        print(f"  📊 Index Stats")        print(f"     Documents: {len(self.documents)}")        print(f"     Dimensions: {emb_array.shape[1]}")        print(f"     Memory: {emb_array.nbytes / 1024:.1f} KB")        print(f"     Model: {self.model}")# Build the search engineengine = SemanticSearchEngine()# Add a knowledge baseknowledge_base = [    "Python is a high-level programming language known for its readability and versatility.",    "Machine learning algorithms learn patterns from data to make predictions.",    "Docker containers package applications with their dependencies for consistent deployment.",    "REST APIs use HTTP methods like GET, POST, PUT, DELETE for web services.",    "PostgreSQL is an open-source relational database with advanced features.",    "Kubernetes orchestrates containerized applications across clusters of machines.",    "React is a JavaScript library for building user interfaces with components.",    "Git version control tracks changes in source code during development.",    "Neural networks are computing systems inspired by biological brain networks.",    "Redis is an in-memory data store used for caching and message brokering.",    "GraphQL is a query language for APIs that gives clients control over data fetching.",    "TensorFlow and PyTorch are popular frameworks for deep learning research and deployment.",    "Microservices architecture breaks applications into small, independently deployable services.",    "OAuth 2.0 is an authorization framework for secure access delegation.",    "WebSocket provides full-duplex communication channels over a single TCP connection.",    "Natural language processing enables computers to understand and generate human language.",    "Apache Kafka is a distributed event streaming platform for high-throughput data pipelines.",    "HTTPS encrypts web traffic using TLS/SSL certificates for secure communication.",    "Agile methodology emphasizes iterative development, collaboration, and rapid delivery.",    "Vector databases store and search high-dimensional embeddings for AI applications.",]metadata = [{"id": i, "category": "tech"} for i in range(len(knowledge_base))]print("🔨 Building Search Index")engine.add_documents(knowledge_base, metadata)engine.stats()

In [ ]:
# Search the knowledge basequeries = [    "How do I build a web API?",    "What's the best way to deploy my application?",    "I need to store data efficiently",    "How can AI understand text?",    "secure my website",]print("🔍 Semantic Search Results")print("=" * 70)for query in queries:    results = engine.search(query, top_k=3)    print(f"\n🔎 Query: \"{query}\"")    for r in results:        relevance = "🟢" if r["score"] > 0.5 else "🟡" if r["score"] > 0.3 else "🔴"        print(f"   {relevance} [{r['score']:.4f}] {r['text'][:80]}...")

In [ ]:
# Edge case: query in a different language (if using multilingual model)print("🌍 Cross-Domain Search Test")print("=" * 50)edge_queries = [    "containerization and devops",     # Should match Docker/K8s    "frontend development framework",  # Should match React    "data pipeline streaming",         # Should match Kafka    "authentication and security",     # Should match OAuth/HTTPS]for q in edge_queries:    results = engine.search(q, top_k=2)    print(f"\n🔎 \"{q}\"")    for r in results:        print(f"   [{r['score']:.4f}] {r['text'][:70]}...")

---## ✅ Notebook 03 Summary| Concept | Key Takeaway ||---------|-------------|| Embeddings | Dense vectors capturing semantic meaning; similar text → nearby vectors || Cosine similarity | Most common metric; range [-1, 1]; direction-based || Dot product | = Cosine for normalized vectors; used by FAISS || Euclidean | Absolute distance; good for anomaly detection || Batch embedding | 10-50x faster than single calls; always batch || Model choice | OpenAI small for general; local (sentence-transformers) for privacy/cost || Curse of dimensionality | Distances converge in high-D but relative ordering is preserved |### ➡️ Next: [Notebook 04 — Basic RAG](./04_basic_rag.ipynb)*Combine embeddings with LLMs to build your first Retrieval-Augmented Generation pipeline.*